---
#### Give Me Some Credit - Credit Risk Classification

---

This notebook demonstrates exploratory data analysis (EDA) and classification modeling using the 'Give Me Some Credit' dataset. 

The goal is to predict the likelihood of serious delinquency (default) within 2 years for loan applicants, using only numeric features. 

Typical EDA steps, data cleaning, and model building are included for practical learning and experimentation.

In [ ]:
# Load and preview the 'Give Me Some Credit' dataset (Kaggle)
import pandas as pd

# Update the path if your file name is different
credit_data_path = r'D:\Makesh\Working\AI\RPS\Day02-8th Nov\datasets\give_me_some_credit\cs-training.csv'

df_credit = pd.read_csv(credit_data_path)

# Remove the 'Unnamed: 0' column if present
df_credit = df_credit.loc[:, df_credit.columns != 'Unnamed: 0']

print('Shape:', df_credit.shape)
df_credit.sample(5)

Shape: (150000, 12)
Shape: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
110496,0,0.251763,55,0,0.264974,10000.0,18,0,2,0,1.0
53027,0,0.062887,63,0,0.476881,4000.0,6,0,0,0,0.0
70027,0,0.029361,35,0,0.328816,4500.0,8,0,1,0,2.0
126600,0,0.115803,48,0,0.147501,2060.0,6,0,0,0,0.0
84162,0,0.413822,77,0,1.115930,1198.0,7,0,1,0,0.0


In [3]:
# Show all column names for reference
print('Column names:')
print(list(df_credit.columns))

Column names:
['SeriousDlqin2yrs', 'RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']


In [4]:
# Rename columns to shorter, more readable names
rename_dict = {
    'SeriousDlqin2yrs': 'target',
    'RevolvingUtilizationOfUnsecuredLines': 'revol_util',
    'age': 'age',
    'NumberOfTime30-59DaysPastDueNotWorse': 'late_30_59',
    'DebtRatio': 'debt_ratio',
    'MonthlyIncome': 'income',
    'NumberOfOpenCreditLinesAndLoans': 'open_credit',
    'NumberOfTimes90DaysLate': 'late_90',
    'NumberRealEstateLoansOrLines': 'real_estate_loans',
    'NumberOfTime60-89DaysPastDueNotWorse': 'late_60_89',
    'NumberOfDependents': 'dependents'
}

df_credit = df_credit.rename(columns=rename_dict)

print('Renamed columns:')
print(list(df_credit.columns))

Renamed columns:
['target', 'revol_util', 'age', 'late_30_59', 'debt_ratio', 'income', 'open_credit', 'late_90', 'real_estate_loans', 'late_60_89', 'dependents']


In [5]:
print('Shape:', df_credit.shape)

Shape: (150000, 11)


#### duplicate rows analysis

In [6]:
# Check for duplicates
num_duplicates = df_credit.duplicated().sum()
print('Number of duplicate rows:', num_duplicates)

Number of duplicate rows: 609


In [7]:
# Get all duplicate rows (including all occurrences)
duplicates = df_credit[df_credit.duplicated(keep=False)]
print(f'Total duplicate rows (all occurrences): {duplicates.shape[0]}')

Total duplicate rows (all occurrences): 960


In [8]:
# Frequency of each duplicate pattern
print('\nFrequency of each duplicate pattern:')
dupe_counts = duplicates.value_counts().reset_index(name='count')
display(dupe_counts.sample(10))


Frequency of each duplicate pattern:


,target,revol_util,age,late_30_59,debt_ratio,income,open_credit,late_90,real_estate_loans,late_60_89,dependents,count
39,0,1.0,24,0,0.0,0.0,1,0,0,0,0.0,2
42,0,1.0,25,0,0.0,764.0,1,0,0,0,0.0,2
30,0,0.0,28,0,0.0,2500.0,2,0,0,0,0.0,2
20,0,0.0,22,0,0.0,0.0,1,0,0,0,0.0,2
18,0,0.0,26,0,0.0,1.0,3,0,0,0,0.0,2
7,0,0.0,22,0,0.0,929.0,2,0,0,0,0.0,5
36,0,1.0,22,0,0.0,0.0,0,0,0,0,0.0,2
10,0,1.0,24,0,0.0,820.0,1,0,0,0,0.0,4
11,0,1.0,22,0,0.0,0.0,1,0,0,0,0.0,3
22,0,0.0,22,0,0.0,929.0,3,0,0,0,0.0,2


In [9]:
# Target value distribution among duplicates
print('\nTarget value counts in duplicates:')
print(duplicates['target'].value_counts())


Target value counts in duplicates:
target
0    928
1     32
Name: count, dtype: int64


In [10]:
# Remove duplicate rows from the DataFrame
df_credit_nodup = df_credit.drop_duplicates()
print(f"Shape after removing duplicates: {df_credit_nodup.shape}")


Shape after removing duplicates: (149391, 11)


#### Missing value analysis

In [ ]:
# Missing value analysis on the cleaned DataFrame (no duplicates)
print('Missing values per column:')
print(df_credit_nodup.isnull().sum())

print('\nPercentage of missing values per column:')
print((df_credit_nodup.isnull().mean() * 100).round(2))


In [11]:
print('\nRows with any missing values:', df_credit_nodup.isnull().any(axis=1).sum())

display(df_credit_nodup[df_credit_nodup.isnull().any(axis=1)].sample(5))


Rows with any missing values: 29221


,target,revol_util,age,late_30_59,debt_ratio,income,open_credit,late_90,real_estate_loans,late_60_89,dependents
128520,1,0.009786,78,0,9.0,NaN,2,0,0,0,NaN
53069,0,0.911823,67,0,5908.0,NaN,25,0,1,0,0.0
10418,0,0.900617,63,2,3231.0,NaN,11,0,2,2,0.0
77987,0,0.054300,28,0,392.0,NaN,4,0,0,0,2.0
89136,0,0.168992,50,0,1166.0,NaN,3,0,1,0,0.0


#### Impact of Missing Values in 'income' and 'dependents' Columns

**Monthly Income ('income'):**
- A key financial feature for credit risk modeling, indicating an applicant’s ability to repay loans.
- Missing values can reduce model predictive power. Imputation (mean/median) may introduce bias if missingness is not random.
- Recommended to analyze if missingness relates to the target or other features. Impute carefully, or use a “missing” indicator.

**Number of Dependents ('dependents'):**
- Reflects financial obligations; more dependents may mean higher expenses and risk.
- Missing values can reduce model accuracy, especially for certain applicant groups.
- Impute with median or zero (if reasonable), or add a “missing” flag.

**General Impact:**
- Systematic missingness (e.g., low-income applicants not reporting income) can bias the model.
- Removing rows with missing values reduces sample size and may affect generalizability.
- Imputation or flagging missingness allows retention of data and can improve model robustness.

*Next steps: Choose an appropriate strategy for handling missing values (imputation, flagging, or removal) based on your analysis and modeling goals.*

**Missing Value Treatment for 'income' Column:**
- **Mean/Median Imputation:** 
    - Median is preferred for income, as income distributions are often skewed by high earners. 
    - Use when missingness is random and the proportion of missing values is moderate to high. 
    - Imputing with median preserves central tendency and reduces the impact of outliers.
- **Zero Imputation:** 
    - Not recommended for income, as zero is not a realistic value and may distort the model. 
    - Only consider if zero has a real-world meaning in your context (rare for income).
- **Model-Based Imputation:** 
    - Use regression or KNN to predict missing income based on other features. 
    - Best when you have enough data and want more accurate imputation. 
    - Use when missingness is not random and other features are predictive of income.
- **Missing Indicator:** 
    - Add a binary column (e.g., 'income_missing') to flag rows where income is missing. 
    - Use when you suspect missingness itself is informative for the target. 
    - Recommended to combine with imputation for robust modeling.
- **Row Removal:** 
    - Only if the number of missing values is very small, otherwise you lose valuable data. 
    - Use when data loss is minimal and you want to avoid imputation bias.
- **Best Practice:** 
    - Use median imputation for 'income' and add a missing indicator column. 
    - This approach is robust, simple, and allows the model to learn from missingness patterns.
- *Choose the strategy based on the proportion and pattern of missingness, the importance of the feature, and your modeling goals.*

In [ ]:
# Deeper analysis of the 'income' column before choosing imputation strategy
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Basic statistics for income (with and without missing values)
print('Income statistics (including missing values):')
print(df_credit_nodup['income'].describe())

In [ ]:
print('\nNumber of missing income values:', df_credit_nodup['income'].isnull().sum())

In [ ]:
# Distribution plot (excluding missing values)
plt.figure(figsize=(8,4))
sns.histplot(df_credit_nodup['income'].dropna(), bins=50, kde=True)
plt.title('Distribution of Monthly Income (excluding missing values)')
plt.xlabel('Monthly Income')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Boxplot to check for outliers
plt.figure(figsize=(8,2))
sns.boxplot(x=df_credit_nodup['income'].dropna())
plt.title('Boxplot of Monthly Income (excluding missing values)')
plt.show()

In [ ]:
# Compare target distribution for missing vs non-missing income
income_missing = df_credit_nodup['income'].isnull()

print('\nTarget distribution for missing income:')
print(df_credit_nodup.loc[income_missing, 'target'].value_counts(normalize=True))

print('\nTarget distribution for non-missing income:')
print(df_credit_nodup.loc[~income_missing, 'target'].value_counts(normalize=True))

**Interpretation of Target Distribution for Missing vs Non-Missing Income**
- For rows with missing income, the proportion of defaults (target=1) is 5.7%.
- For rows with non-missing income, the proportion of defaults is 7.0%.

**What does this mean?**
- The default rate is lower among applicants with missing income values compared to those who reported income.
- This suggests missingness in the income column is not random; applicants who did not report income are less likely to default.
- Possible reasons: Applicants with missing income may be more financially stable, or there may be reporting bias (e.g., retirees, self-employed, or high-income individuals not disclosing income).

**Modeling Implication:**
- Missingness in income is informative and should not be ignored.
- Adding a missing indicator column will help the model capture this pattern.
- Imputation (e.g., median) can be used, but the missing indicator is essential to avoid losing this signal.

**Note:**
- Using a missing indicator does not mean you should exclude the 'income' column.
- Best practice is to use both the imputed 'income' values (e.g., fill missing with median) and the 'income_missing' indicator column together.
- The model will learn from both the actual income values and the pattern of missingness, improving predictive power.
- Keep the 'income' column (with imputed values) and add a 'income_missing' column.

In [ ]:
# Missing value treatment for 'income' column
df_credit_nodup['income_missing'] = df_credit_nodup['income'].isnull().astype(int)
income_median                     = df_credit_nodup['income'].median()

df_credit_nodup['income']         = df_credit_nodup['income'].fillna(income_median)

print(f"Filled missing 'income' values with median: {income_median}")
print("'income_missing' indicator column added.")

df_credit_nodup[['income', 'income_missing']].sample(5)

#### Missing value treatment for 'dependents' column

In [ ]:
# Deeper analysis of the 'dependents' column before choosing imputation strategy
import matplotlib.pyplot as plt
import seaborn as sns

# Basic statistics for dependents (with and without missing values)
print('Dependents statistics (including missing values):')
print(df_credit_nodup['dependents'].describe())

print('\nNumber of missing dependents values:', df_credit_nodup['dependents'].isnull().sum())

# Distribution plot (excluding missing values)
plt.figure(figsize=(8,4))
sns.histplot(df_credit_nodup['dependents'].dropna(), bins=20, kde=False)
plt.title('Distribution of Number of Dependents (excluding missing values)')
plt.xlabel('Number of Dependents')
plt.ylabel('Frequency')
plt.show()

# Boxplot to check for outliers
plt.figure(figsize=(8,2))
sns.boxplot(x=df_credit_nodup['dependents'].dropna())
plt.title('Boxplot of Number of Dependents (excluding missing values)')
plt.show()

# Compare target distribution for missing vs non-missing dependents
dependents_missing = df_credit_nodup['dependents'].isnull()
print('\nTarget distribution for missing dependents:')
print(df_credit_nodup.loc[dependents_missing, 'target'].value_counts(normalize=True))

print('\nTarget distribution for non-missing dependents:')
print(df_credit_nodup.loc[~dependents_missing, 'target'].value_counts(normalize=True))

**Dependents Column Analysis & Interpretation**
- The distribution of 'dependents' is right-skewed, with most applicants having few dependents.
- Outliers may exist (e.g., unusually high number of dependents), but the median is a robust measure for imputation.
- The number of missing values is relatively small compared to the dataset size.

**Target Distribution:**
- If the default rate (target=1) among rows with missing 'dependents' is similar to or lower than those with non-missing values, missingness may not be strongly associated with risk.
- If the default rate is higher for missing 'dependents,' missingness could be informative and should be flagged.

**Modeling Implication:**
- Impute missing 'dependents' with the median to preserve central tendency and avoid bias from outliers.
- Add a 'dependents_missing' indicator column to help the model learn if missingness itself is predictive.

In [ ]:
df_credit_nodup['dependents_missing'] = df_credit_nodup['dependents'].isnull().astype(int)

dependents_median               = df_credit_nodup['dependents'].median()
df_credit_nodup['dependents']   = df_credit_nodup['dependents'].fillna(dependents_median)

print(f"Filled missing 'dependents' values with median: {dependents_median}")
print("'dependents_missing' indicator column added.")
df_credit_nodup[['dependents', 'dependents_missing']].sample(5)

#### after the missing val treament

In [ ]:
df_credit_nodup.sample(10)

#### More feature engineering to follow...

**Why Check for Multicollinearity?**
- Multicollinearity occurs when two or more predictor variables are highly correlated.
- It can negatively impact model interpretability and the stability of coefficient estimates, especially for linear models like logistic regression.

**Benefits of Checking Multicollinearity:**
- Identifies redundant features that may not add value to the model.
- Prevents inflated standard errors and unreliable model coefficients.
- Guides feature selection, improving model generalizability.

**How to Check for Multicollinearity:**
- Calculate the correlation matrix for all numeric features.
- Use Variance Inflation Factor (VIF) to quantify multicollinearity.
- Drop or combine highly correlated features if necessary.


In [ ]:
# Check for multicollinearity: correlation matrix and VIF
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Select features for multicollinearity check (exclude target)
features = [col for col in df_credit_nodup.columns if col != 'target']
X = df_credit_nodup[features]

# Correlation matrix
corr_matrix = X.corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Feature Correlation Matrix')
plt.show()

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print('Variance Inflation Factor (VIF) for each feature:')
display(vif_data.sort_values('VIF', ascending=False))

**VIF Results and Interpretation:**

- VIF values above 10 indicate severe multicollinearity. The features `late_60_89`, `late_90`, and `late_30_59` are highly collinear.
- High multicollinearity can destabilize model coefficients and reduce interpretability, especially for linear models.

**Recommended Actions:**
- Consider removing or combining the highly collinear features (`late_60_89`, `late_90`, `late_30_59`).
- Recalculate VIF after removal to confirm multicollinearity is reduced.
- Retain features with low VIF (typically < 5) as they do not pose multicollinearity issues.

#### How to Choose a Scaler for Feature Scaling
- **StandardScaler (Z-score):** Use if features are approximately normally distributed. Centers data to mean 0 and standard deviation 1. Good for linear models (e.g., logistic regression, SVM).
- **MinMaxScaler:** Use if features are not normally distributed or have outliers. Scales data to a fixed range (usually 0 to 1). Useful for algorithms sensitive to feature magnitude (e.g., neural networks, k-NN).
- **RobustScaler:** Use if data contains many outliers. Uses median and interquartile range, making it robust to outliers.
- **Normalizer:** Use if you want to scale each sample (row) to unit norm, mainly for text or sparse data.

**How to Decide:**
- Plot feature distributions (histograms, boxplots).
- If most features are normal: StandardScaler.
- If features have outliers: RobustScaler.
- If features are on different scales but bounded: MinMaxScaler.

*Visualize your feature distributions before choosing a scaler. The right choice improves model performance and convergence.*

In [ ]:
# Plot feature distributions for all numeric columns (excluding target)
import matplotlib.pyplot as plt
import seaborn as sns

numeric_features = [col for col in df_credit_nodup.columns if col != 'target']

plt.figure(figsize=(16, 12))
for i, col in enumerate(numeric_features):
    plt.subplot(4, 3, i+1)
    sns.histplot(df_credit_nodup[col], bins=30, kde=True)
    plt.title(col)
    plt.xlabel('')
    plt.ylabel('')
plt.tight_layout()
plt.suptitle('Feature Distributions', y=1.02)
plt.show()

#### Skewness Correction for Numeric Features

**Understanding Skewness and Its Impact on Machine Learning**
- **Skewness** measures how much a distribution deviates from being symmetric around its mean.
- If a feature is **right-skewed** (positive skew), most values are clustered at the lower end, with a long tail to the right (higher values).
- If a feature is **left-skewed** (negative skew), most values are at the higher end, with a long tail to the left (lower values).
- **Intuitive Example:** Income is often right-skewed: most people earn average amounts, but a few earn much more, stretching the distribution.

**How Skewness Affects Machine Learning:**
- Many ML algorithms (especially linear models) assume features are normally distributed (symmetric, bell-shaped).
- Highly skewed features can bias model training, making it harder for algorithms to learn patterns.
- Skewness can cause issues with feature scaling, outlier sensitivity, and convergence during training.
- Transforming skewed features (e.g., with log1p) helps stabilize variance, reduce outlier impact, and improve model accuracy and interpretability.
- Less skewed features lead to better performance for models that rely on statistical assumptions, such as logistic regression and SVM.

**Simple Examples: Skewness and log1p Transformation**
- **Right-Skewed Example:**
  - Imagine a dataset of incomes: [30k, 35k, 40k, 45k, 50k, 200k]
  - Most values are low, but one value (200k) is much higher, creating a long right tail.
  - This is right-skewed: the mean is pulled up by the large value.

- **Left-Skewed Example:**
  - Imagine test scores: [95, 96, 97, 98, 99, 60]
  - Most values are high, but one value (60) is much lower, creating a long left tail.
  - This is left-skewed: the mean is pulled down by the small value.

- **log1p Transformation Example:**
  - For the income data above:
    - Original: [30,000, 35,000, 40,000, 45,000, 50,000, 200,000]
    - log1p: [10.31, 10.46, 10.60, 10.71, 10.82, 12.21] (using natural log)
    - The transformed values are closer together, reducing the effect of the outlier.

- **Why Use log1p?**
  - log1p compresses large values and spreads out small values, making the distribution more symmetric.
  - It helps models learn better by reducing the impact of extreme values and stabilizing variance.

In [ ]:
# Show original skewness of selected features before log transformation
skewed_cols       = ['income', 'revol_util', 'debt_ratio', 'late_60_89', 'late_90', 'late_30_59', 'real_estate_loans', 'dependents', 'open_credit']
original_skewness = df_credit_nodup[skewed_cols].skew().sort_values(ascending=False)
print('Original skewness before log1p transformation:')
display(original_skewness)

In [ ]:
# Apply log1p transformation to highly skewed features and visualize results
skewed_cols         = ['income', 'revol_util', 'debt_ratio', 'late_60_89', 'late_90', 'late_30_59', 'real_estate_loans', 'dependents', 'open_credit']
df_credit_nodup_log = df_credit_nodup.copy()

for col in skewed_cols:
    # log1p handles zeros safely
    df_credit_nodup_log[col + '_log1p'] = np.log1p(df_credit_nodup_log[col])

# Plot distributions after log1p transformation
plt.figure(figsize=(16, 12))
for i, col in enumerate(skewed_cols):
    plt.subplot(4, 3, i+1)
    sns.histplot(df_credit_nodup_log[col + '_log1p'], bins=30, kde=True)
    plt.title(col + ' (log1p)')
    plt.xlabel('')
    plt.ylabel('')
plt.tight_layout()
plt.suptitle('Log1p-Transformed Feature Distributions', y=1.02)
plt.show()

# Check skewness after transformation
skewness_log = df_credit_nodup_log[[col + '_log1p' for col in skewed_cols]].skew().sort_values(ascending=False)
print('Skewness after log1p transformation:')
display(skewness_log)

**Scaler Recommendation After Log1p Transformation**
- After applying the log1p transformation, most features have reduced skewness and distributions closer to normal.
- This makes the StandardScaler (Z-score normalization) a suitable choice for scaling, especially for linear models like logistic regression or SVM.

**Why StandardScaler?**
- StandardScaler centers data to mean 0 and standard deviation 1.
- Works best when features are approximately normal, which is achieved after log1p transformation.
- Improves model convergence and interpretability for algorithms sensitive to feature scaling.

**When to Consider RobustScaler?**
- If some features still show significant outliers after transformation, RobustScaler can be used for those specific columns.
- RobustScaler uses the median and interquartile range, making it robust to outliers.

**Next Step:**
- Apply StandardScaler to the log1p-transformed features for model training.
- If needed, use RobustScaler for columns with remaining outliers.

**What is log1p?**
- `log1p(x)` computes the natural logarithm of (1 + x), i.e., $\log(1 + x)$.
- It is mathematically defined as:
  $$ \text{log1p}(x) = \ln(1 + x) $$
- This transformation is useful for reducing skewness in data, especially when features have a wide range or contain zeros.
- Unlike the standard logarithm, log1p can safely handle zero values (since $\log(1 + 0) = 0$), avoiding undefined results.
- Commonly used in machine learning to normalize highly skewed features and improve model performance.

**Feature Scaling Strategy**
- After log1p transformation, most features are closer to normal distribution.
- Use StandardScaler to standardize log1p-transformed features (mean=0, std=1).
- This improves model convergence and performance for algorithms sensitive to feature scaling (e.g., logistic regression, SVM, neural networks).
- If any features still have strong outliers, consider RobustScaler for those columns.

**Steps:**
1. Select log1p-transformed feature columns.
2. Fit StandardScaler on training data and transform.
3. Use the scaled features for modeling.

**Note:**
- Missing indicator columns (e.g., 'income_missing_log1p', 'dependents_missing_log1p') are binary and should not be scaled.
- Only scale continuous numeric log1p-transformed features.

In [ ]:
# Apply StandardScaler to log1p-transformed features (excluding missing indicators)
from sklearn.preprocessing import StandardScaler
continuous_cols = [col + '_log1p' for col in ['income', 'revol_util', 'debt_ratio', 'late_60_89', 'late_90', 'late_30_59', 'real_estate_loans', 'dependents', 'open_credit']]
indicator_cols  = ['income_missing', 'dependents_missing']

scaler = StandardScaler()
scaled_features  = scaler.fit_transform(df_credit_nodup_log[continuous_cols])

df_credit_scaled = pd.DataFrame(scaled_features, columns=continuous_cols)

# Reset index to ensure alignment
df_credit_scaled = df_credit_scaled.reset_index(drop=True)
df_credit_nodup_log = df_credit_nodup_log.reset_index(drop=True)

for col in indicator_cols:
    df_credit_scaled[col] = df_credit_nodup_log[col]

df_credit_scaled['target'] = df_credit_nodup_log['target']

In [ ]:
df_credit_scaled.sample(10)

#### Model selection and evaluation

**Top Machine Learning Models for Credit Risk Classification**

Given the nature of the columns (mostly numeric, some binary indicators, all scaled/normalized), here are the top models to consider:

1. **Logistic Regression**
   - Interpretable, robust to multicollinearity (if addressed), works well with standardized numeric and binary features.
   - Good baseline for binary classification.

2. **Random Forest**
   - Handles both numeric and binary features, robust to outliers and non-linear relationships.
   - Provides feature importance and is less sensitive to scaling.

3. **Gradient Boosting Machines (e.g., XGBoost, LightGBM, CatBoost)**
   - Excellent for tabular data, handles missing values and various feature types.
   - Often achieves state-of-the-art results in credit risk and Kaggle competitions.

4. **Support Vector Machine (SVM)**
   - Effective with standardized features, can model complex decision boundaries.
   - May be slower on large datasets, but good for well-prepared numeric data.

5. **K-Nearest Neighbors (KNN)**
   - Simple, non-parametric, works well with scaled features.
   - Useful for benchmarking, but less interpretable and slower on large datasets.

**Recommendation:**
- Start with Logistic Regression and KNN as baselines.
- RF and SVM are strong contenders for this dataset.
- Try Gradient Boosting (XGBoost/LightGBM/CatBoost) for best performance.

In [ ]:
# K-Nearest Neighbors (KNN) Classifier: Train/Test Split Example
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Prepare features and target
feature_cols = [
    'income_log1p', 'revol_util_log1p', 'debt_ratio_log1p', 'late_60_89_log1p', 'late_90_log1p',
    'late_30_59_log1p', 'real_estate_loans_log1p', 'open_credit_log1p',
    'income_missing', 'dependents_missing'
 ]
X = df_credit_scaled[feature_cols]
y = df_credit_scaled['target']

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [ ]:
# Initialize and fit KNN classifier
knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

In [ ]:
# Predict and evaluate
y_pred = knn.predict(X_test)

In [ ]:
acc = accuracy_score(y_test, y_pred)

print(f'KNN Test Accuracy: {acc:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

#### Hyperparameter Search: Randomized vs Full Grid Search
- **RandomizedSearchCV**: Samples a fixed number of parameter settings from specified distributions. Faster for large grids, good for initial exploration.
- **GridSearchCV**: Exhaustively tests all combinations in the parameter grid. More thorough, but slower for large grids.
- Both methods use cross-validation to find the best KNN hyperparameters. Results below show best parameters and test accuracy for each approach.

In [ ]:
# RandomizedSearchCV for KNN (random grid search)
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

In [ ]:
knn = KNeighborsClassifier()

param_dist = {
    'n_neighbors':  np.arange(2, 20),
    'weights':      ['uniform', 'distance'],
    'p':            [1, 2]
}

In [ ]:
random_search = RandomizedSearchCV(knn, 
                                   param_distributions  =param_dist, 
                                   n_iter               =10, 
                                   cv                   =5, 
                                   scoring              ='accuracy', 
                                   n_jobs               =-1, 
                                   random_state         =42)

random_search.fit(X_train, y_train)

print('RandomizedSearchCV Best parameters:', random_search.best_params_)
print(f'RandomizedSearchCV Best cross-validated accuracy: {random_search.best_score_:.4f}')

In [ ]:
best_knn_random = random_search.best_estimator_
y_pred_random   = best_knn_random.predict(X_test)

print(f'RandomizedSearchCV KNN Test Accuracy: {accuracy_score(y_test, y_pred_random):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_random))

#### Logistic Regression

In [ ]:
# Logistic Regression: Train and evaluate
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
logreg.fit(X_train, y_train)
y_pred_lr = logreg.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(f'Logistic Regression Test Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_lr))

In [ ]:
# Naive Bayes: Train and evaluate
from sklearn.naive_bayes import GaussianNB
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)
print(f'Naive Bayes Test Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_nb))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_nb))

In [ ]:
# Random Forest: Train and evaluate
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print(f'Random Forest Test Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_rf))

#### SMOTE Oversampling  

In [ ]:
#!pip install imbalanced-learn

In [ ]:
# Apply SMOTE to balance the classes in the training set
from imblearn.over_sampling import SMOTE

# SMOTE should only be applied to the training set to avoid data leakage
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('Original training set class distribution:')
print(y_train.value_counts())
print('\nSMOTE training set class distribution:')
print(pd.Series(y_train_smote).value_counts())

In [ ]:
# Logistic Regression with SMOTE-balanced training data
from sklearn.linear_model import LogisticRegression
logreg_smote = LogisticRegression(max_iter=1000, random_state=42)
logreg_smote.fit(X_train_smote, y_train_smote)
y_pred_lr_smote = logreg_smote.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(f'Logistic Regression (SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_lr_smote):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr_smote))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_lr_smote))

In [ ]:
# Naive Bayes with SMOTE-balanced training data
from sklearn.naive_bayes import GaussianNB
nb_smote = GaussianNB()
nb_smote.fit(X_train_smote, y_train_smote)
y_pred_nb_smote = nb_smote.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(f'Naive Bayes (SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_nb_smote):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_nb_smote))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_nb_smote))

#### Boosting Models: XGBoost, LightGBM, CatBoost

In [ ]:
# XGBoost with SMOTE-balanced training data
#!pip install xgboost
from xgboost import XGBClassifier
xgb_smote = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_smote.fit(X_train_smote, y_train_smote)
y_pred_xgb_smote = xgb_smote.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(f'XGBoost (SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_xgb_smote):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb_smote))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_xgb_smote))

In [ ]:
# Hyperparameter tuning for XGBoost with SMOTE-balanced training data
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2, 0.3],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
random_search_xgb = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=20, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)
random_search_xgb.fit(X_train_smote, y_train_smote)

print('Best XGBoost parameters:', random_search_xgb.best_params_)
print(f'Best cross-validated accuracy: {random_search_xgb.best_score_:.4f}')

In [ ]:
best_xgb = random_search_xgb.best_estimator_
y_pred_xgb_opt = best_xgb.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(f'XGBoost (Optimal Params, SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_xgb_opt):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb_opt))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_xgb_opt))

#### Neural Network & Linear Classifiers: MLPClassifier and SGDClassifier (scikit-learn)

We now evaluate two additional classifiers:

- **MLPClassifier**: A feedforward neural network (multi-layer perceptron) suitable for tabular data. Sensitive to feature scaling and hyperparameters.
- **SGDClassifier**: Linear classifier (e.g., logistic regression, SVM) trained with stochastic gradient descent. Supports class weights and regularization.

Both models are trained on SMOTE-balanced data for fair comparison. Results include accuracy, classification report, and confusion matrix.

#### Simple Perceptron (No SMOTE)
We now train a basic linear Perceptron classifier using the original training and test sets (no resampling). This provides a baseline for linear separability and minority class recall without oversampling.

In [ ]:
# Simple Perceptron (no SMOTE) on original train/test split
from sklearn.linear_model import Perceptron
perceptron = Perceptron(max_iter=1000, tol=1e-3, random_state=42)
perceptron.fit(X_train, y_train)
y_pred_perc = perceptron.predict(X_test)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print(f'Perceptron (No SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_perc):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_perc))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_perc))

#### MLPClassifier with Early Stopping and Training Loss Curve (No SMOTE)
We now train an MLPClassifier with early stopping enabled, using the original training and test sets. The training loss curve is plotted to visualize convergence and detect overfitting.

In [ ]:
# MLPClassifier with early stopping and training loss curve (no SMOTE)
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt

mlp_es = MLPClassifier(hidden_layer_sizes=(64, 32, 16, 8, 4), 
                       max_iter=300, 
                       random_state=42, 
                       early_stopping=True, 
                       validation_fraction=0.1, 
                       n_iter_no_change=5, 
                       verbose=True)

mlp_es.fit(X_train, y_train)
#mlp_es.fit(X_train_smote, y_train_smote)

y_pred_mlp_es = mlp_es.predict(X_test)

# Plot training loss curve
plt.figure(figsize=(8,4))
plt.plot(mlp_es.loss_curve_, marker='o')
plt.title('MLPClassifier Training Loss Curve (Early Stopping)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f'MLPClassifier (Early Stopping, No SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_mlp_es):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_mlp_es))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_mlp_es))

#### SGDClassifier (No SMOTE)
We now train and evaluate an SGDClassifier (stochastic gradient descent linear model) using the original training and test sets. This provides a baseline for linear models with different loss functions and regularization.

In [ ]:
# SGDClassifier (no SMOTE) on original train/test split
from sklearn.linear_model import SGDClassifier
sgd = SGDClassifier(loss='log_loss', max_iter=1000, tol=1e-3, random_state=42, early_stopping=True, validation_fraction=0.1, n_iter_no_change=5, verbose=True)
sgd.fit(X_train, y_train)
y_pred_sgd = sgd.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f'SGDClassifier (No SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_sgd):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_sgd))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_sgd))

**Next Steps for Improving Recall and Model Performance:**
- Adjust decision threshold for classifiers to improve recall for class 1 (defaults).
- Try alternative resampling methods (SMOTEENN, SMOTETomek, ADASYN) or ensemble resampling.
- Engineer new features, combine existing ones, or remove highly collinear features.
- Analyze feature importance to focus on the most predictive features.



#### Threshold Tuning for Improved Recall
Adjusting the decision threshold for classifiers can improve recall for the minority class (defaults). Instead of the default 0.5 threshold, we can select a threshold that increases recall for class 1. Below, we plot recall, precision, and F1-score for different thresholds and select the optimal value.

In [ ]:
mlp_es = MLPClassifier(hidden_layer_sizes=(64, 32, 16, 8, 4), 
                       max_iter=300, 
                       random_state=42, 
                       early_stopping=True, 
                       validation_fraction=0.1, 
                       n_iter_no_change=5, 
                       verbose=True)

mlp_es.fit(X_train, y_train)
#mlp_es.fit(X_train_smote, y_train_smote)

y_pred_mlp_es = mlp_es.predict(X_test)

# Plot training loss curve
plt.figure(figsize=(8,4))
plt.plot(mlp_es.loss_curve_, marker='o')
plt.title('MLPClassifier Training Loss Curve (Early Stopping)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print(f'MLPClassifier (Early Stopping, No SMOTE) Test Accuracy: {accuracy_score(y_test, y_pred_mlp_es):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_mlp_es))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_mlp_es))

In [ ]:
# Define thresholds and compute recall, precision, F1-score for MLPClassifier (mlp_es)
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.linspace(0.05, 1, 10)
recalls    = []
precisions = []
f1s        = []

y_scores_mlp = mlp_es.predict_proba(X_test)[:, 1]

for thresh in thresholds:
    y_pred_thresh = (y_scores_mlp >= thresh).astype(int)
    recalls.append(recall_score(y_test, y_pred_thresh))
    precisions.append(precision_score(y_test, y_pred_thresh))
    f1s.append(f1_score(y_test, y_pred_thresh))

plt.figure(figsize=(10,6))
plt.plot(thresholds, recalls, label='Recall', color='blue')
plt.plot(thresholds, precisions, label='Precision', color='green')
plt.plot(thresholds, f1s, label='F1-score', color='red')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Recall, Precision, F1-score vs. Threshold (MLPClassifier, early stopping)')
plt.legend()
plt.grid(True)
plt.show()

# Find threshold with highest recall (or F1-score)
best_idx = np.argmax(recalls)
best_thresh = thresholds[best_idx]
print(f'Best threshold for recall: {best_thresh:.2f} (Recall: {recalls[best_idx]:.3f}, Precision: {precisions[best_idx]:.3f}, F1: {f1s[best_idx]:.3f})')

In [ ]:
# Select threshold with best F1-score for MLPClassifier (mlp_es) and print metrics
from sklearn.metrics import classification_report, confusion_matrix

best_f1_idx = np.argmax(f1s)
best_f1_thresh = thresholds[best_f1_idx]
print(f'Best threshold for F1-score: {best_f1_thresh:.2f} (Recall: {recalls[best_f1_idx]:.3f}, Precision: {precisions[best_f1_idx]:.3f}, F1: {f1s[best_f1_idx]:.3f})')
y_pred_best_f1 = (y_scores_mlp >= best_f1_thresh).astype(int)
print('\nClassification Report (Best F1 Threshold):')
print(classification_report(y_test, y_pred_best_f1))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_best_f1))

# Compare with recall-maximizing threshold
print(f'Best threshold for recall: {best_thresh:.2f} (Recall: {recalls[best_idx]:.3f}, Precision: {precisions[best_idx]:.3f}, F1: {f1s[best_idx]:.3f})')
y_pred_best_recall = (y_scores_mlp >= best_thresh).astype(int)
print('\nClassification Report (Best Recall Threshold):')
print(classification_report(y_test, y_pred_best_recall))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_best_recall))